In [1]:
import pandas as pd
import numpy as np
import faiss
import requests

In [2]:
index = faiss.read_index('index')

In [4]:
df = pd.read_csv('alza_dataset.csv')

In [8]:
def create_textual_representation(row):
    subcategory = f"\n\nSubcategory: {row['breadcrumb_subcategory']}" if pd.notna(row["breadcrumb_subcategory"]) else ""
    textual_representation = f"""Product name: {row["name"]},

Price: {row["price"]},

Description: {row["spec_summary"]}

Category: {row["breadcrumb_category"]}{subcategory}


"""
    return textual_representation

df['textual_representation'] = df.apply(create_textual_representation, axis=1)

In [14]:
prompt = "photography"
def get_matches(user_prompt):
    res = requests.post('http://localhost:11434/api/embeddings',
                        json={
                            'model': 'nomic-embed-text',
                            'prompt': user_prompt
                        }
                       )
    
    embedding = np.array([res.json()['embedding']], dtype = 'float32')
    
    D, I = index.search(embedding, 10)
    
    best_matches = np.array(df['textual_representation'])[I.flatten()]
    
    for match in best_matches[1:]: # skip the first (it's identical to the query)
        print('NEXT PROPERTY')
        print(match)
        print()
get_matches(prompt)

NEXT PROPERTY
Product name: Canon PFI-1000Y Yellow,

Price: 1289.0,

Description: Cartridge - yellow, for Canon imagePROGRAF PRO-1000, up to 3365 photographs of a format 10x15cm and up to 329 A2 photos, 80ml

Category: Printers and Scanners

Subcategory: Toners & Inks




NEXT PROPERTY
Product name: Canon PFI-1000C Cyan,

Price: 1289.0,

Description: Cartridge - Cyan, for Canon imagePROGRAF PRO-1000, up to 5025 pages of photographs format 10x15cm photos or 675 A2, 80 ml

Category: Printers and Scanners

Subcategory: Toners & Inks




NEXT PROPERTY
Product name: Canon PFI-1000PCI Cyan,

Price: 1289.0,

Description: Cartridge - Photo Cyan, for Canon imagePROGRAF PRO-1000, up to 5140 pages of photographs format 10 x 15cm photos or 600 A2, 80ml

Category: Printers and Scanners

Subcategory: Toners & Inks




NEXT PROPERTY
Product name: Rollei Compact Traveler No.1 Karbon - orange,

Price: 2590.0,

Description: Tripod - for digital camera, spherical head, transport height of 33cm, extended 

# Discussion
It seems so work better with this data than with the real estate dataset, probably because the data here is more structured and is mostly in English.

In [15]:
prompt = "perfume"
get_matches(prompt)

NEXT PROPERTY
Product name: SANYTOL Disinfectant air freshener floral scent 300 ml,

Price: 83.01,

Description: Air Freshener -  spray - manual, will keep your interior fresh for hours 

Category: Candles and Incense Sticks

Subcategory: Air Fresheners




NEXT PROPERTY
Product name: DIFFUSIL Repellent Family 100 ml,

Price: 139.0,

Description: Repellent - perfume free, repels ticks, mosquitoes 

Category: Insect and Pest Protection

Subcategory: Repellents




NEXT PROPERTY
Product name: OFF! Protect Spray 100 ml,

Price: 109.0,

Description: Repellent - perfumed, repels ticks, mosquitoes 

Category: Insect and Pest Protection

Subcategory: Repellents




NEXT PROPERTY
Product name: SONAX PROFILINE Skin Care, 1L,

Price: 1131.0,

Description: Car Upholstery Cleaner - perfumed and reviving, volume 1000 ml

Category: Car Cosmetics

Subcategory: Interior




NEXT PROPERTY
Product name: FROSCH toilet gel lavender 750 ml,

Price: 63.75,

Description: Toilet Cleaner - floral fragrance, ch

# Discussion
The first maatch doesn't have "perfume" word in it. This proves that the model has some intelligence and it is not 
doing some primitive regex search.
However in the second and last match we can clearly see "perfume free".

Embedding models encode semantic meaning into vector space, and "perfume free" products live in a similar semantic neighbourhood to "perfume" — they're products in the same category, described using the same domain vocabulary. A repellent labeled "perfume free" and actual perfume share a lot of contextual signal.
Let's try tp prompt more explicitely.

In [16]:
prompt = "body fragrance perfume eau de toilette personal scent"
get_matches(prompt)

NEXT PROPERTY
Product name: COCCOLINO Primavera fragrance for wardrobe 3 pcs,

Price: 78.4,

Description: Closet Fragrance to the wardrobe

Category: Candles and Incense Sticks

Subcategory: Air Fresheners




NEXT PROPERTY
Product name: SANYTOL Disinfectant air freshener mountain scent 300 ml,

Price: 83.0,

Description: Air Freshener -  spray - manual, will keep your interior fresh for hours 

Category: Candles and Incense Sticks

Subcategory: Air Fresheners




NEXT PROPERTY
Product name: FROSCH toilet gel lavender 750 ml,

Price: 63.75,

Description: Toilet Cleaner - floral fragrance, chlorine-free, eco-friendly, volume 750 ml 

Category: Cleaning Products

Subcategory: Toilet




NEXT PROPERTY
Product name: ATTITUDE Lemon scented bathroom cleaner with sprayer 800 ml,

Price: 149.0,

Description: Bathroom Cleaner - for bathroom, 

Category: Cleaning Products

Subcategory: Bathroom




NEXT PROPERTY
Product name: FROSCH Oase aroma diffuser Lavender 90 ml,

Price: 226.9,

Description

In [17]:
prompt = "iphone"
get_matches(prompt)

NEXT PROPERTY
Product name: Screenshield APPLE iPhone 8 Plus - display,

Price: 99.0,

Description: Film Screen Protector for mobile phone Apple iPhone 8 Plus, on the screen of the device 

Category: Phone Accessories

Subcategory: Screen Protectors




NEXT PROPERTY
Product name: Simba My first smartphone,

Price: 189.0,

Description: Kids’ Phone , suitable for children from 1 year 

Category: Electronic and Robotic Toys

Subcategory: Electronics for Kids




NEXT PROPERTY
Product name: FIXED Sarif horizontal XXL Black,

Price: 449.0,

Description: Phone Case -, flip case, artificial leather, anti-scratch screen protector, cutouts for connectors and buttons, handsewn and system for attaching to belt, maximum device height: 137mm, maximum device width: 70mm 

Category: Phone Accessories

Subcategory: Cases




NEXT PROPERTY
Product name: Screenshield SAMSUNG G960 Galaxy S9 full body protector,

Price: 349.0,

Description: Film Screen Protector for mobile phone SAMSUNG Galaxy S9, on dis

In [19]:
prompt = "something to read"
get_matches(prompt)

NEXT PROPERTY
Product name: Cabin Porn - Cottages at the End of the World,

Price: 489.0,

Description: Book - picture publication for all lovers of hunting, nature and also architecture - author Klein Zach, 328 pages, Czech, solid without glossy cover

Category: Books

Subcategory: Hobby




NEXT PROPERTY
Product name: Journalistic English: A reader for intermediate learners,

Price: 299.0,

Description: Kniha 

Category: Books

Subcategory: Textbooks




NEXT PROPERTY
Product name: Přehled světové literatury,

Price: 60.71,

Description: Kniha 

Category: Books

Subcategory: Textbooks




NEXT PROPERTY
Product name: Diary of a Wimpy Kid book 1,

Price: 169.0,

Description: Kniha 

Category: Books

Subcategory: In Foreign Languages




NEXT PROPERTY
Product name: Slovník korejské literatury,

Price: 439.0,

Description: Kniha 

Category: Books




NEXT PROPERTY
Product name: ORY Books Farm animals,

Price: 69.0,

Description: Leporello - children's book, 6 pages

Category: Books




N

# Discussion
The examples show that the embedding model has intelligence and recommendation system is not just word matching.
Next step is to explore strategies that could enhance recommendation quality.